In [2]:
# --- HuggingFace auth (gated repos: meditron-7b, Llama-3.1) ---
# Paste your token from https://huggingface.co/settings/tokens (read scope).
# Requires having clicked "Agree and access repository" on the model page first.
import os
from huggingface_hub import login

HF_TOKEN = os.environ.get("HF_TOKEN", "")   # or hardcode: "hf_..."
if HF_TOKEN:
    login(token=HF_TOKEN)
    print("HF login OK")
else:
    print("[WARN] no HF_TOKEN set — gated repos will fail with 401")

HF login OK


In [3]:
# -*- coding: utf-8 -*-
"""demo_sensitivity_runner.py

Phase 1 generation runs for the Demonstration-Sensitivity study.
Base: healthslm_eval_fewshot_fixed.py — inherits its corrected field mappings
(meddialog dialogue_context+patient_message / doctor_response, mtsamples
transcription/description), context-window detection + tail truncation,
and chat-template support.

PROMPT COMPATIBILITY: zero-shot prompts == format_generation_prompt of the
base script; few-shot prompts are built EXACTLY like the base script's
format_demonstration / build_few_shot_prefix (full demo prompt incl.
instruction + gold answer after 'Response:', joined by DEMO_SEP='\\n\\n---\\n\\n',
MAX_DEMO_CHARS=2000 per-demo truncation). New runs are therefore directly
comparable to the existing few-shot k=5 rows in the results sheet.
The ONLY intended differences: demos come from the frozen pool=100 with
seeds 0-4 (instead of one hash-based draw from the full train split), and
the LOCKED alphabetical ordering rule is applied after sampling.

Per model x task x test instance -> 6 generations:
  R1  zero              : 1  (same prompt as healthslm_eval_fixed.py)
  R2  random_seed{0..4} : 5  (k=5 demos from a fixed pool, seeds 0-4,
                              LOCKED ordering rule applied)

Retriever axis (R3, dyn_{retriever}) is a LATER step — deliberately excluded
from this script. The demo pool (pool=30, POOL_SEED=42) is built and saved
here so the future retriever runs use the exact same pool.

Frozen parameters (Phase 0, locked 2026-08-17):
  k=5 | K=5 seeds 0-4, fixed draws shared across all test questions |
  pool=30 (POOL_SEED=42; some train splits <100) | N=full official test split |
  ordering rule = Li et al. 2025 default ordering (generation tasks):
      sort all demos alphabetically by demo input text (lowercase, lstrip) |
  greedy T=0 | demo placement = start of prompt.

Output: one JSONL per model x task x condition with FULL generation text and
complete config metadata. No efficiency logging. Scoring (S1-S4) is a
separate pass.

Run (one model per run):
    pip install vllm
    python demo_sensitivity_runner.py
"""

import os
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

import json
import time
import random
import hashlib

import torch
assert torch.cuda.is_available(), "GPU not found."

from transformers import AutoTokenizer, AutoConfig

# ------------------------------------------------------------------------
# 0. FROZEN CONFIG  (Phase 0 deliverable — committed before generation)
# ------------------------------------------------------------------------

MODEL_NAME        = "epfl-llm/meditron-7b"   # <- change per run (5-model roster)
USE_CHAT_TEMPLATE = False                         # match healthslm_eval_fixed.py
MODEL_SLUG = MODEL_NAME.split('/')[-1].lower().replace('-', '_').replace('.', '_')

CONFIG = {
    "model_name": MODEL_NAME,
    "use_chat_template": USE_CHAT_TEMPLATE,
    "tasks": ["aci_bench", "meddialog", "medicationqa", "mtsamples", "mtsamples_proc"],
    "k": 5,
    "seeds": [0, 1, 2, 3, 4],
    "pool_size": 30,   # some train splits are <100 examples; 30 fits all tasks
    "pool_seed": 42,
    "ordering_rule": "alphabetical_by_demo_input_text_lower",  # Li et al. 2025 default ordering
    "temperature": 0.0,
    "demo_placement": "start_of_prompt",
    "demo_sep": "\n\n---\n\n",       # same as healthslm_eval_fewshot_fixed.py
    "max_demo_chars": 2000,          # same as healthslm_eval_fewshot_fixed.py
    "config_version": "2026-08-17",
}

DEMO_SEP       = CONFIG['demo_sep']
MAX_DEMO_CHARS = CONFIG['max_demo_chars']

BASE_DIR = '/workspace/biomedical_datasets_combined'
OUT_DIR  = '/workspace/demo_sensitivity_runs'
os.makedirs(OUT_DIR, exist_ok=True)
with open(os.path.join(OUT_DIR, 'config_frozen.json'), 'w') as f:
    json.dump(CONFIG, f, indent=2)

# ------------------------------------------------------------------------
# 1. Task file discovery — needs BOTH test and train files
#    (healthslm_eval_fixed.py's should_skip drops 'train'; demo pools need it)
# ------------------------------------------------------------------------

# task -> normalization kind (mtsamples_proc uses the mtsamples field mappings,
# same as detect_dataset_kind's longest-match routing in the base script)
TASK_KIND = {
    'aci_bench': 'aci_bench', 'meddialog': 'meddialog',
    'medicationqa': 'medicationqa', 'mtsamples': 'mtsamples',
    'mtsamples_proc': 'mtsamples',
}


# task -> substring that identifies its files (matched on the lowercased
# filename with '_'/'-' stripped). aci_bench files are named like
# 'aci_all_test_sample...', so the key is just 'aci'.
TASK_FILE_KEY = {
    'aci_bench':      'aci',
    'meddialog':      'meddialog',
    'medicationqa':   'medicationqa',
    'mtsamples':      'mtsamples',
    'mtsamples_proc': 'mtsamples',
}


def find_task_files(base_dir, task):
    """Return (test_path, train_path). A file belongs to the task if the
    TASK_FILE_KEY substring is in its name; 'test'/'train' in the name picks
    the split; mtsamples vs mtsamples_proc is disambiguated by 'proc'."""
    test_path, train_path = None, None
    key = TASK_FILE_KEY[task]
    for root, _, files in os.walk(base_dir):
        if 'checkpoint' in root.lower():        # skip .ipynb_checkpoints dirs
            continue
        for fname in files:
            if not fname.endswith(('.jsonl', '.json')):
                continue
            if 'checkpoint' in fname.lower():   # skip Jupyter autosave copies
                continue
            low = fname.lower().replace('_', '').replace('-', '')
            if key not in low:
                continue
            if ('proc' in task) != ('proc' in low):
                continue
            if 'test' in low:
                test_path = os.path.join(root, fname)
            elif 'train' in low:
                train_path = os.path.join(root, fname)
    return test_path, train_path


def load_rows(path):
    if path.endswith('.jsonl'):
        with open(path, encoding='utf-8') as f:
            return [json.loads(l) for l in f if l.strip()]
    with open(path, encoding='utf-8') as f:
        data = json.load(f)
        return data if isinstance(data, list) else [data]

# ------------------------------------------------------------------------
# 2. Normalization — copied verbatim in behavior from healthslm_eval_fixed.py
#    for the 5 generation kinds (incl. FIX #3 meddialog and mtsamples fixes)
# ------------------------------------------------------------------------

def _coerce_str(v):
    if v is None:
        return ''
    if isinstance(v, (list, tuple)):
        return '\n'.join(_coerce_str(x) for x in v)
    if isinstance(v, dict):
        return '\n'.join(f"{k}: {_coerce_str(val)}" for k, val in v.items())
    return str(v)


def _first_nonempty(d, *keys):
    for k in keys:
        if k in d and d[k] not in (None, ''):
            return d[k]
    return ''


def normalize_example(ex, kind):
    norm = {'question': '', 'context': '', 'reference': '', '_kind': kind}

    if kind == 'aci_bench':
        norm['question']  = "Generate the structured clinical note for the dialogue above."
        norm['context']   = _coerce_str(_first_nonempty(
            ex, 'dialogue', 'src', 'conversation', 'input', 'transcript'))
        norm['reference'] = _coerce_str(_first_nonempty(
            ex, 'note', 'tgt', 'summary', 'reference', 'output', 'gold'))

    elif kind == 'mtsamples':
        # FIX: transcription -> context, description -> reference
        norm['context']   = _coerce_str(_first_nonempty(
            ex, 'transcription', 'input', 'src', 'dialogue', 'context'))
        norm['reference'] = _coerce_str(_first_nonempty(
            ex, 'description', 'sample_name', 'output', 'summary', 'tgt',
            'note', 'reference', 'gold'))

    elif kind == 'meddialog':
        # FIX #3: merge dialogue_context + patient_message; ref = doctor_response
        _dial_ctx = _coerce_str(ex.get('dialogue_context', ''))
        _pat_msg  = _coerce_str(ex.get('patient_message', ''))
        _merged   = f"{_dial_ctx}\nPatient: {_pat_msg}" if (_dial_ctx and _pat_msg) else ''
        norm['context']   = _merged or _coerce_str(_first_nonempty(
            ex, 'src', 'dialogue', 'utterances', 'conversation', 'description',
            'input', 'context', 'patient_query', 'query',
            'dialogue_context', 'patient_message'))
        norm['reference'] = _coerce_str(_first_nonempty(
            ex, 'tgt', 'response', 'utterance', 'summary', 'output',
            'reference', 'answer', 'label', 'doctor_reply', 'reply',
            'doctor_response'))

    elif kind == 'medicationqa':
        norm['question']  = ex.get('question', ex.get('Question', ''))
        norm['context']   = _coerce_str(_first_nonempty(
            ex, 'context', 'Focus (Drug)', 'Focus', 'focus'))
        norm['reference'] = _coerce_str(_first_nonempty(ex, 'answer', 'Answer', 'reference'))

    else:
        raise ValueError(kind)
    return norm


def demo_input_text(norm):
    """The 'demo input text' used by the ordering rule and the retrievers."""
    return f"{norm['context']}\n{norm['question']}".strip()

# ------------------------------------------------------------------------
# 3. Prompt building — copied VERBATIM in behavior from
#    healthslm_eval_fewshot_fixed.py (format_generation_prompt,
#    format_demonstration, build_few_shot_prefix) so new runs match the
#    existing zero-shot and few-shot k=5 rows in the results sheet.
# ------------------------------------------------------------------------

DATASET_INSTRUCTIONS = {
    'aci_bench': (
        "Summarize the conversation to generate a clinical note with four sections:\n"
        "1. HISTORY OF PRESENT ILLNESS\n2. PHYSICAL EXAM\n3. RESULTS\n4. ASSESSMENT AND PLAN\n\n"
        "The conversation is:"
    ),
    'mtsamples':    "Given various information about a patient, return a reasonable treatment plan for the patient.",
    'meddialog':    "Generate a one sentence summary of this patient-doctor conversation.",
    'medicationqa': "Please answer the following consumer health question.",
}
# aci_bench (underscore) is NOT MedHELM-style in the base script — its
# instruction lives in `question` and it uses the Context:/Task: layout.
MEDHELM_STYLE = {'meddialog', 'medicationqa', 'mtsamples'}


def chat_wrap(s):
    if USE_CHAT_TEMPLATE:
        try:
            return tokenizer.apply_chat_template(
                [{"role": "user", "content": s.strip()}],
                tokenize=False, add_generation_prompt=True)
        except Exception:
            return f"<s>[INST] {s.strip()} [/INST]"
    return s.strip()


def format_generation_prompt(norm):
    """VERBATIM logic from healthslm_eval_fewshot_fixed.py."""
    ctx = norm.get('context', '')
    q   = norm.get('question', '')
    if norm['_kind'] in MEDHELM_STYLE:
        instr = DATASET_INSTRUCTIONS.get(norm['_kind'], '')
        parts = []
        if instr: parts.append(instr)
        if ctx:   parts.append(ctx)
        if q and (not instr or q != instr): parts.append(q)
        return chat_wrap("\n\n".join(parts) + "\n\nResponse:")
    body = ""
    if ctx: body += f"Context: {ctx}\n"
    if q:   body += f"Task: {q}\n"
    return chat_wrap(body + "\nResponse:")


def format_demonstration(demo_norm):
    """VERBATIM from the base script: full prompt (incl. instruction) + gold
    answer after 'Response:'; per-demo truncation at MAX_DEMO_CHARS keeping
    the last 300 chars so 'Response:' survives."""
    prompt = format_generation_prompt(demo_norm)
    answer = str(demo_norm.get('reference', '')).strip()
    if not answer:
        return ''
    if len(prompt) + len(answer) > MAX_DEMO_CHARS:
        keep_tail = 300
        prompt = prompt[:MAX_DEMO_CHARS - keep_tail - len(answer)] + " ... " + prompt[-keep_tail:]
    return f"{prompt} {answer}"


def build_few_shot_prefix(demos):
    """VERBATIM: demos joined by DEMO_SEP; trailing DEMO_SEP before the test prompt."""
    rendered = [format_demonstration(d) for d in demos]
    rendered = [r for r in rendered if r]
    return DEMO_SEP.join(rendered) + DEMO_SEP if rendered else ""


def build_prompt(kind, demos, test_norm):
    """demos = normalized examples ALREADY in final order; [] = zero-shot.
    full_prompt = demo_prefix + test_prompt, exactly like the base script."""
    test_prompt = format_generation_prompt(test_norm)
    prefix = build_few_shot_prefix(demos)
    return prefix + test_prompt if prefix else test_prompt

# ------------------------------------------------------------------------
# 4. Demo pool, LOCKED ordering rule, seeded draws
# ------------------------------------------------------------------------

def build_pool(train_rows, kind):
    """Fixed pool of pool_size examples; pool_id = row index in the train file."""
    normed = []
    for i, ex in enumerate(train_rows):
        n = normalize_example(ex, kind)
        if demo_input_text(n) and n['reference'].strip():
            normed.append({'pool_id': i, **n})
    rng = random.Random(CONFIG['pool_seed'])
    if len(normed) > CONFIG['pool_size']:
        normed = rng.sample(normed, CONFIG['pool_size'])
    return normed


def apply_ordering_rule(demos):
    """LOCKED 2026-08-17 — Li et al. 2025 default ordering, generation tasks:
    sort all demos alphabetically by demo input text (lowercase, lstrip)."""
    return sorted(demos, key=lambda d: demo_input_text(d).lstrip().lower())


def seeded_draws(pool, k, seeds):
    """Fixed draws: one draw per seed, SHARED across all test questions."""
    return {s: apply_ordering_rule(random.Random(s).sample(pool, k)) for s in seeds}

# ------------------------------------------------------------------------
# 5. vLLM — context-window handling copied from healthslm_eval_fixed.py
# ------------------------------------------------------------------------

from vllm import LLM, SamplingParams

print(f"Loading tokenizer for {MODEL_NAME} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
_mcfg    = AutoConfig.from_pretrained(MODEL_NAME, trust_remote_code=True)
_raw     = getattr(_mcfg, 'max_position_embeddings', None) \
           or getattr(tokenizer, 'model_max_length', None) or 4096
MODEL_MAX_LEN = int(_raw) if _raw and int(_raw) < 100_000 else 4096
print(f"Model max length: {MODEL_MAX_LEN}")

llm = LLM(model=MODEL_NAME, dtype="auto", trust_remote_code=True,
          gpu_memory_utilization=0.90, enforce_eager=False,
          enable_chunked_prefill=True, max_model_len=MODEL_MAX_LEN)

_GEN_MAX_OUT = min(1024, MODEL_MAX_LEN // 2)
SP = SamplingParams(temperature=CONFIG['temperature'], max_tokens=_GEN_MAX_OUT)
CONFIG['max_tokens'] = _GEN_MAX_OUT


def truncate_prompts_to_context(prompts):
    """Tail-keep truncation (same as base script). NOTE for few-shot: tail-keep
    drops the EARLIEST demos first and preserves the test instance + 'Response:'
    line. Truncation events are counted and stored per record."""
    limit = MODEL_MAX_LEN - _GEN_MAX_OUT - 8
    result, truncated = [], []
    for p in prompts:
        ids = tokenizer.encode(p, add_special_tokens=False)
        if len(ids) > limit:
            p = tokenizer.decode(ids[-limit:], skip_special_tokens=False)
            truncated.append(True)
        else:
            truncated.append(False)
        result.append(p)
    if any(truncated):
        print(f"  [WARN] truncated {sum(truncated)}/{len(prompts)} prompts "
              f"(kept last {limit} tokens — earliest demos dropped first)")
    return result, truncated


# ------------------------------------------------------------------------
# 6. Run one condition = one batch, one JSONL (FULL text, full metadata)
# ------------------------------------------------------------------------

def run_condition(task, condition, prompts, meta_per_instance):
    out_path = os.path.join(OUT_DIR, f"gen_{MODEL_SLUG}_{task}_{condition}.jsonl")
    if os.path.exists(out_path):
        print(f"  [skip] exists: {os.path.basename(out_path)}")
        return

    prompts, was_truncated = truncate_prompts_to_context(prompts)

    t0 = time.perf_counter()
    outputs = llm.generate(prompts, SP)
    wall = time.perf_counter() - t0

    n_hit_cap = sum(1 for o in outputs if len(o.outputs[0].token_ids) >= _GEN_MAX_OUT)
    if n_hit_cap > len(outputs) * 0.1:
        print(f"  [WARN] {n_hit_cap}/{len(outputs)} outputs hit max_tokens={_GEN_MAX_OUT}")

    with open(out_path, 'w', encoding='utf-8') as f:
        for i, o in enumerate(outputs):
            rec = {
                'model': MODEL_NAME,
                'task': task,
                'condition': condition,          # zero | random_seed{0-4}
                'k': 0 if condition == 'zero' else CONFIG['k'],
                'config_version': CONFIG['config_version'],
                'ordering_rule': None if condition == 'zero' else CONFIG['ordering_rule'],
                'prompt_truncated': was_truncated[i],
                'prompt_sha1': hashlib.sha1(prompts[i].encode()).hexdigest(),
                'output_text': o.outputs[0].text,    # FULL text — required for S2/S3
                **meta_per_instance[i],
            }
            f.write(json.dumps(rec, ensure_ascii=False) + '\n')

    print(f"  [done] {condition}: {len(prompts)} gens, {wall:.0f}s "
          f"-> {os.path.basename(out_path)}")

# ------------------------------------------------------------------------
# 7. Main loop
# ------------------------------------------------------------------------

for task in CONFIG['tasks']:
    kind = TASK_KIND[task]
    print(f"\n=== {task} (kind={kind}) ===")
    test_path, train_path = find_task_files(BASE_DIR, task)
    assert test_path,  f"no test file found for {task}"
    assert train_path, f"no TRAIN file found for {task} — the demo pool needs the train split"
    print(f"  test:  {os.path.basename(test_path)}\n  train: {os.path.basename(train_path)}")

    test_norm = [normalize_example(ex, kind) for ex in load_rows(test_path)]
    pool = build_pool(load_rows(train_path), kind)
    print(f"  N={len(test_norm)} test instances, pool={len(pool)}")

    # Save the pool so the future retriever-axis runs reuse the EXACT same pool
    pool_path = os.path.join(OUT_DIR, f'pool_{task}.json')
    if not os.path.exists(pool_path):
        with open(pool_path, 'w', encoding='utf-8') as f:
            json.dump(pool, f, ensure_ascii=False, indent=1)

    empty_refs = sum(1 for t in test_norm if not t['reference'].strip())
    if empty_refs:
        print(f"  [WARN] {empty_refs}/{len(test_norm)} empty references — check field names")

    base_meta = [{'instance_id': i, 'reference': t['reference']}
                 for i, t in enumerate(test_norm)]

    # R1 zero-shot (prompt identical to healthslm_eval_fixed.py)
    prompts = [build_prompt(kind, [], t) for t in test_norm]
    meta = [{**m, 'seed': None, 'retriever': None, 'demo_pool_ids': []} for m in base_meta]
    run_condition(task, 'zero', prompts, meta)

    # R2 random-draw axis: K=5 fixed draws, LOCKED ordering rule
    for s, demos in seeded_draws(pool, CONFIG['k'], CONFIG['seeds']).items():
        demo_ids = [d['pool_id'] for d in demos]
        prompts = [build_prompt(kind, demos, t) for t in test_norm]
        meta = [{**m, 'seed': s, 'retriever': None, 'demo_pool_ids': demo_ids}
                for m in base_meta]
        run_condition(task, f'random_seed{s}', prompts, meta)

print("\nAll conditions done. Scoring (S1-S4) runs as a separate pass over the JSONLs.")


Loading tokenizer for epfl-llm/meditron-7b ...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/344 [00:00<?, ?B/s]

...

WARNING 08-20 12:27:18 scheduler.py:1555] Sequence group 12503 is preempted by PreemptionMode.RECOMPUTE mode because there is not enough KV cache space. This can affect the end-to-end performance. Increase gpu_memory_utilization or tensor_parallel_size to provide more KV cache memory. total_num_cumulative_preemption=5551


Processed prompts:  76%|███████▌  | 380/500 [05:25<04:53,  2.44s/it, est. speed input: 1186.23 toks/s, output: 1194.23 toks/s]

WARNING 08-20 12:28:29 scheduler.py:1555] Sequence group 12563 is preempted by PreemptionMode.RECOMPUTE mode because there is not enough KV cache space. This can affect the end-to-end performance. Increase gpu_memory_utilization or tensor_parallel_size to provide more KV cache memory. total_num_cumulative_preemption=5601


Processed prompts: 100%|██████████| 500/500 [06:40<00:00,  1.25it/s, est. speed input: 1268.90 toks/s, output: 1277.45 toks/s]


  [WARN] 500/500 outputs hit max_tokens=1024
  [done] random_seed4: 500 gens, 402s -> gen_meditron_7b_mtsamples_proc_random_seed4.jsonl

All conditions done. Scoring (S1-S4) runs as a separate pass over the JSONLs.


In [1]:
!pip install -U typing_extensions

In [2]:
!pip install "pydantic<2.10" && echo restart kernel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 20.0 MB/s  0:00:00
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.46.4
    Uninstalling pydantic_core-2.46.4:
      Successfully uninstalled pydantic_core-2.46.4
  Attempting uninstall: pydantic
    Found existing installation: pydantic 2.13.4
    Uninstalling pydantic-2.13.4:
      Successfully uninstalled pydantic-2.13.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pydantic]1/2 [pydantic]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mistral-common 1.11.7 requires numpy<2.4,>=1.25; python_version <= "3.12", but you have numpy 2.4.6 which is incompatible.
vllm 0.6.6 requires numpy<2.0.0, but you have numpy 2.4.6 which is incompatible.
restart kernel


In [ ]:
pip install -U typing_extensions